# S7 · When optimization goes wrong: reward hacking, length and entropy

Study path stage 7 of 7 · Chapter 14 · Day 24 · **DGX Spark GPU first**

Every earlier stage trained a policy to raise a reward. This stage asks what happens when the reward
is wrong. A reward model is a **proxy**: a learned estimate of the quality we care about, accurate on
answers like its training data and unreliable elsewhere. Optimizing it hard moves the policy to
exactly the answers where it is unreliable. By the end you will be able to compute the
overoptimization curve of a known reward exactly, explain why best-of-n and the KL-regularized optimum
spend KL differently, remove a linear length channel from a reward, measure entropy collapse,
compare mean and min ensembles, and write a diagnostic that flags the classic warning signs in a
training log, then apply it to a real Qwen3-0.6B GRPO run against a reward model you train here.

| Part | Question | Discussions |
|---|---|---|
| A · Overoptimization, exactly | If π maximizes E[proxy] − β·KL exactly, how do proxy and gold reward change as β falls? | [D3, D35](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
| B · Best-of-n | How much KL does best-of-n spend, and does it overoptimize too? | [D3, D13](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
| C · Length hacking | What does a reward that credits length do to the optimum, and how is the length channel removed? | [D2, D31](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
| D · Entropy | What does entropy collapse look like in a policy-gradient run, and what do a KL penalty and an entropy bonus change? | [D35, glossary](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
| E · Mitigations | KL strength, reward-model ensembles (mean vs min), length control: which gold outcomes improve, and which errors remain? | [D3, D13](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
| F · Diagnostics | Which training-log patterns should make you stop and read samples? | plan P2 |
| G · Qwen3 | Does a 1,000-pair reward model reward length, and does a short GRPO run exploit it with β = 0 vs β > 0? | [D13, D31, D35](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |

**How the exercises work.** Each exercise has the same rhythm:

1. **Read the task**, then write your lines where you see `...` in the exercise cell.
2. **Run the check.** It tests your code on small cases and prints ✅ or ❌ with a hint. Nothing
   crashes if you haven't finished; you'll see ⏳ instead.
3. **Compare with the reference**, which follows each check and then runs the next experiment.

Later cells use your version if its check passed, otherwise the reference, via
`mine_or_ref(globals(), …)`, so one stuck exercise never blocks the rest. Predict cells ask you to
write an answer before running the reference.

**Data.** Toy parts: synthetic answer sets generated in the notebook. Part G: Anthropic HH-RLHF
`helpful-base` (MIT license), 1,000 training pairs for the reward model, 300 held-out test pairs,
and single-turn user questions from `train[20000:]` (never seen by the reward model) as RL prompts.

**Environment.** Kernel `Python (DGX Spark Native)` with CUDA, the Qwen3-0.6B weights and the HH
dataset in the local Hugging Face cache; nothing is downloaded. Checks come from the tested module
`dongxi_llms.study_path.s7_failure_modes`.

**Run time.** About 9 minutes on the Spark: under a minute for Parts A–F, about 3 minutes for the
reward model and about 5 minutes for the two GRPO runs; peak GPU memory about 23 GiB.

**Scale and limits.** The toy parts are exact for their finite answer sets: every toy number is a
property of a stated reward, not an estimate. They show mechanisms (Gao et al. 2022's rise-then-fall
of gold reward, best-of-n's KL, length channels, ensemble behaviour); they do not reproduce Gao et
al.'s scaling laws, which used reward models up to 3B parameters and a 6B "gold" model. Part G is a
small real run: 30 GRPO iterations of 16 answers each. It verifies the measurement pipeline and
reports what moved beyond noise; it is not evidence that a policy became more or less helpful.

In [ ]:
import inspect, math, random, sys, time
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dongxi_llms").is_dir())
sys.path.insert(0, str(ROOT / "src"))               # the course package, wherever this copy lives
import matplotlib.pyplot as plt
import numpy as np
import torch, torch.nn.functional as F
from dongxi_llms.study_path import qwen as Q
from dongxi_llms.study_path import s7_failure_modes as s7
from dongxi_llms.study_path.common import mine_or_ref

assert torch.cuda.is_available(), "Use the Python (DGX Spark Native) kernel: this notebook is GPU first."
DEVICE = "cuda"
random.seed(0); np.random.seed(0); torch.manual_seed(0)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, GRAY, GREEN, RED = "#2563EB", "#EA580C", "#94A3B8", "#16A34A", "#DC2626"
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

---

# Part A · Overoptimization, computed exactly

RLHF trains the policy on the **KL-regularized objective**

$$J(\pi) = \mathbb{E}_{y\sim\pi}[R(y)] - \beta\,\mathrm{KL}(\pi\,\|\,\pi_{\mathrm{ref}}),$$

where $R$ is the reward model's score and $\pi_{\mathrm{ref}}$ is the starting (SFT) policy. Over a
finite answer set, $J$ has an exact maximizer (D35; derived in S2):

$$\pi_\beta(y) = \frac{\pi_{\mathrm{ref}}(y)\,e^{R(y)/\beta}}{Z_\beta},\qquad Z_\beta = \sum_{y'} \pi_{\mathrm{ref}}(y')\,e^{R(y')/\beta}.$$

Large $\beta$ keeps $\pi_\beta$ at $\pi_{\mathrm{ref}}$; small $\beta$ concentrates it on the
highest-reward answers and spends more KL. Sweeping $\beta$ therefore traces the path an optimizer
takes as it moves further from the reference, with no sampling noise and no optimizer: what we see
is a property of the reward itself.

Two rewards matter. The **gold reward** $g$ is the quality we actually want (in Gao et al. 2022 a
6B reward model stands in for human judgment; here it is a formula). The **proxy reward** $r$ is
what the optimizer sees, the trained reward model. **Overoptimization** is optimizing $r$ past the
point where $g$ stops improving.

**The answer pool.** $K = 2^{20}$ candidate answers to one prompt. Each answer has a helpfulness
score $h$ and a length score $z$, independent $\mathcal{N}(0,1)$ draws; its length in tokens is
$\ell = 60\,e^{0.5z}$. $\pi_{\mathrm{ref}}$ is uniform over the pool. The rewards are

$$g = h - a\,(z^2 - 1),\qquad r = h + b\,z,\qquad a = 0.5.$$

The gold reward values helpfulness and penalizes lengths far from typical, in either direction. The
proxy sees helpfulness correctly but also credits length linearly with slope $b$, the way the Day 15
reward model's score correlated +0.40 with length (D31). Near $\pi_{\mathrm{ref}}$ the two agree
well; far from it they do not.

In [ ]:
pool = s7.answer_pool()                          # h, z, length [K]; logp_ref = log(1/K)
gold = s7.gold_reward(pool, a=0.5)
proxy = s7.proxy_reward(pool, b=1.0)
K = pool["h"].shape[0]
print(f"K = {K:,} answers | length: median {pool['length'].median():.0f} tokens, "
      f"5–95% [{pool['length'].quantile(.05):.0f}, {pool['length'].quantile(.95):.0f}]")
print(f"under π_ref: E[gold] = {gold.mean():+.3f}, E[proxy] = {proxy.mean():+.3f}, "
      f"corr(proxy, gold) = {torch.corrcoef(torch.stack([proxy, gold]))[0, 1]:.3f}")

The correlation is $1/\sqrt{3} \approx 0.577$: $\mathrm{Cov}(r, g) = \mathrm{Var}(h) = 1$ (because
$\mathrm{Cov}(z, z^2) = 0$), $\mathrm{Var}(r) = 1 + b^2 = 2$ and $\mathrm{Var}(g) = 1 + 2a^2 = 1.5$.
A reward model that ranks answers this well on its own data is a reasonable proxy. The question is
what happens when we optimize it.

### Prediction

## Exercise 1 · the KL-regularized optimum

Return $\log \pi_\beta$ for log-probabilities `logp_ref` (shape `[K]`), rewards `R` (`[K]`) and a
scalar `beta`. One line: add the log-reference and the scaled reward, then normalize with
`torch.log_softmax(..., -1)`. Working in log space avoids overflow: $e^{R/\beta}$ is $e^{100}$ at
$R = 5, \beta = 0.05$.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_tilt(logp_ref, R, beta):
    return ...                                      # log π_β, normalized over the last dimension

In [ ]:
ex1_ok = s7.check_tilt(globals().get("my_tilt"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def tilt_ref(logp_ref, R, beta):
    return torch.log_softmax(logp_ref + R / beta, -1)

s7.check_tilt(tilt_ref, strict=True)
tilt = mine_or_ref(globals(), "my_tilt", ex1_ok, tilt_ref)
for beta in (10.0, 1.0, 0.1):
    p = tilt(pool["logp_ref"], proxy, beta).exp()
    print(f"β = {beta:5.1f}: largest single-answer probability {p.max():.2e} "
          f"(π_ref gives {1 / K:.2e}) | E[proxy] = {(p * proxy).sum():+.3f} | E[gold] = {(p * gold).sum():+.3f}")

### Reference explanation

### Reference answer and mechanism

`log_softmax(logp_ref + R/β)` is $\log\pi_{\mathrm{ref}} + R/\beta - \log Z_\beta$: the
normalizer is the log-sum-exp, computed stably. Dividing by β matters: a small β multiplies the
reward differences, so the distribution sharpens. At β = 10 the policy is close to uniform; at
β = 0.1 a handful of answers carry most of the probability.

Already the printout shows the pattern of this stage: from β = 1 to β = 0.1 the proxy keeps rising
while the gold reward drops.

### Prediction

## Exercise 2 · exact KL between two categorical distributions

Return $\mathrm{KL}(\pi\,\|\,\pi_{\mathrm{ref}}) = \sum_y \pi(y)\,[\log\pi(y) - \log\pi_{\mathrm{ref}}(y)]$
from two log-probability tensors, summing over the last dimension. One line. The weights are
$\pi$, the first argument (D35's worked example gives 0.0851 nats).

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_kl(logp, logp_ref):
    return ...                                      # Σ_y π(y)·(log π(y) − log π_ref(y))

In [ ]:
ex2_ok = s7.check_kl(globals().get("my_kl"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def kl_ref(logp, logp_ref):
    return (logp.exp() * (logp - logp_ref)).sum(-1)

s7.check_kl(kl_ref, strict=True)
kl = mine_or_ref(globals(), "my_kl", ex2_ok, kl_ref)
print(f"maximum possible KL from a uniform π_ref over K answers: log K = {math.log(K):.2f} nats")

### Reference explanation

### Reference answer and mechanism

KL weights the log-ratio by $\pi$, so it measures how surprising the new policy's own answers are
under the reference. Reversing the arguments gives $\mathrm{KL}(\pi_{\mathrm{ref}}\|\pi)$, a
different number (0.0920 instead of 0.0851 in D35's example). With a uniform reference,
$\mathrm{KL} = \log K - H(\pi)$, so it can never exceed $\log K = 13.86$ nats; that ceiling is
reached when the policy puts all its mass on one answer.

### The Gaussian limit: a formula for the whole curve

For a large pool, $\pi_{\mathrm{ref}}$ is close to the standard normal density over $(h, z)$.
Tilting a standard normal by $e^{t\,(h + b z)}$ with $t = 1/\beta$ shifts its mean and keeps its
covariance (complete the square in the exponent):

$$\mathcal{N}(x; 0, I)\, e^{t\,w^\top x} \;\propto\; \mathcal{N}(x;\, t w,\, I),\qquad w = (1, b).$$

So under $\pi_\beta$, $h \sim \mathcal{N}(t, 1)$ and $z \sim \mathcal{N}(bt, 1)$, which gives

$$\mathbb{E}[r] = t(1 + b^2),\qquad \mathbb{E}[g] = t - a\,b^2 t^2,\qquad \mathrm{KL} = \tfrac12\,\|t w\|^2 = \tfrac12\,t^2(1 + b^2).$$

The proxy is linear in $t$ and keeps rising. The gold reward is a downward parabola in $t$: the
helpfulness gain is linear, and the length penalty grows quadratically because the policy keeps
lengthening its answers.

### Prediction

## Predict 1 · where does the gold reward peak?

With $a = 0.5$ and $b = 1$, use the formulas above. At which $t$ (equivalently β = 1/t) is
$\mathbb{E}[g]$ largest? How much KL has the policy spent there, and what is the gold value? Then
predict what the exact computation over the finite pool will print.

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
betas = torch.logspace(1, -0.9, 40, dtype=torch.float64)      # β from 10 down to ≈ 0.13
curves = {}
for b in (1.0, 0.85, 0.7):
    r_b = s7.proxy_reward(pool, b)
    curves[b] = s7.sweep(tilt, kl, pool["logp_ref"], r_b, betas, {"gold": gold, "proxy": r_b,
                                                                  "length": pool["length"]})
a, b = 0.5, 1.0
t_star = 1 / (2 * a * b * b)
print(f"theory (K → ∞): t* = {t_star:.2f}, β* = {1 / t_star:.2f}, KL* = {t_star**2 * (1 + b*b) / 2:.2f} nats, "
      f"gold* = {1 / (4 * a * b * b):.3f}")
c = curves[1.0]
i = int(c["gold"].argmax())
print(f"finite pool:    best β on the grid {c['beta'][i]:.2f}, KL {c['kl'][i]:.2f} nats, gold {c['gold'][i]:.3f}, "
      f"proxy {c['proxy'][i]:.3f}")
exact = tilt(pool["logp_ref"], proxy, 1.0)
print(f"finite pool at β = 1 exactly: KL {kl(exact, pool['logp_ref']):.3f}, gold {s7.expect(exact, gold):.3f}")
j = int((c["kl"] - 4).abs().argmin())
print(f"at KL ≈ {c['kl'][j]:.1f} nats: proxy {c['proxy'][j]:+.2f} (still rising), gold {c['gold'][j]:+.3f} "
      f"(back to the reference's 0); mean length {c['length'][j]:.0f} tokens vs {pool['length'].mean():.0f} under π_ref")

### Reference explanation

### Reference answer and mechanism

Setting $\frac{d}{dt}(t - a b^2 t^2) = 1 - 2ab^2t = 0$ gives $t^* = 1/(2ab^2) = 1$, so
**β\* = 1, KL\* = 1 nat and gold\* = 0.5**. The finite pool agrees to within its sampling error
(its gold at β = 1 is about 0.48; near the top the curve is flat, so the grid's best β can sit a
little to either side). By KL ≈ 4 nats the gold reward is back at the reference's value of zero,
while the proxy has doubled and the mean answer is roughly three times as long.

The mechanism: the KL-regularized optimum moves every feature the proxy credits, in proportion to
the credit. Helpfulness and length both move linearly in $t$. The proxy charges nothing for length,
the gold reward charges quadratically, so the length term eventually dominates.

In terms of $d = \sqrt{\mathrm{KL}}$ the gold curve is $\alpha d - \gamma d^2$ with
$\alpha = \sqrt{2/(1+b^2)}$ and $\gamma = 2ab^2/(1+b^2)$. That is the functional form Gao et al.
(2022) fitted to best-of-n sampling, $d(\alpha - \beta d)$; for RL with PPO they fitted
$d(\alpha - \beta\log d)$. The toy reproduces the shape, not their numbers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
KL_MAX = 6.0                                       # beyond this the finite pool's tail is too thin
c = curves[1.0]
m = c["kl"] <= KL_MAX
d = c["kl"][m].sqrt()
axes[0].plot(d, c["proxy"][m], color=ORANGE, label="proxy r (what the optimizer sees)")
axes[0].plot(d, c["gold"][m], color=BLUE, label="gold g (what we want)")
th = s7.gaussian_theory(1.0)
keep = th["kl"] <= KL_MAX
axes[0].plot(th["kl"][keep].sqrt(), th["gold"][keep], color=BLUE, linestyle=":", label="gold, Gaussian theory")
axes[0].axvline(1.0, color=GRAY, linestyle="--", linewidth=1)
axes[0].text(1.04, 3.0, "predicted gold peak\nKL = 1 nat", color=GRAY, fontsize=8)
axes[0].axhline(0, color="black", linewidth=0.6)
axes[0].set(xlabel="√KL(π_β ‖ π_ref)  (nats^½)", ylabel="expected reward", title="b = 1: proxy keeps rising, gold peaks")
axes[0].legend(fontsize=8, loc="upper left")
for b, color in zip((1.0, 0.85, 0.7), (RED, ORANGE, GREEN)):
    c = curves[b]
    m = c["kl"] <= KL_MAX
    axes[1].plot(c["kl"][m].sqrt(), c["gold"][m], color=color, label=f"b = {b} (finite pool)")
    th = s7.gaussian_theory(b)
    keep = th["kl"] <= KL_MAX
    axes[1].plot(th["kl"][keep].sqrt(), th["gold"][keep], color=color, linestyle=":", linewidth=1)
axes[1].axhline(0, color="black", linewidth=0.6)
axes[1].set(xlabel="√KL(π_β ‖ π_ref)", ylabel="E[gold]", title="smaller length credit b: later, higher peak")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
for b in (1.0, 0.85, 0.7):
    c = curves[b]; m = c["kl"] <= KL_MAX; i = int(c["gold"][m].argmax())
    print(f"b = {b}: gold peak (KL ≤ {KL_MAX:.0f}) {c['gold'][m][i]:.3f} at KL {c['kl'][m][i]:.2f} nats "
          f"(theory {1 / (4 * 0.5 * b * b):.3f} at {(1 + b*b) / (8 * 0.25 * b**4):.2f})")

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Gold and proxy reward of the KL-regularized optimum against √KL, and gold for three length credits b](../figures/chapter-14/study-path-s7_failure_modes-01.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

Left: the shape Gao et al. reported. The proxy reward rises monotonically with KL; the gold reward
rises, peaks and falls, and the gap between the two is the reward model's error on the answers the
policy now prefers. Right: a reward model with a smaller length credit $b$ (a better proxy) peaks
later and higher, $\mathrm{KL}^* = (1+b^2)/(8a^2b^4)$ and $g^* = 1/(4ab^2)$. Gao et al. found the
analogous trend in reward-model size: larger proxy reward models reached higher gold peaks and
overoptimized less at a given KL. The dotted Gaussian curves and the finite pool agree up to the plotted
range. Beyond KL ≈ 6 nats the finite pool runs out of answers far enough in the tail: the tilt
puts most of its mass on a few dozen answers, whose values are sampling accidents (and the KL can
never exceed $\log K$).

The KL penalty sits in this picture as a choice of where to stop: each β picks one point on the
curve. Choosing β is choosing a KL budget. That is why D35 says KL "keeps the policy where the
reward model is valid": the reward model was trained on answers near $\pi_{\mathrm{ref}}$, and β
limits how far from them the policy may go.

---

# Part B · Best-of-n: selection also spends KL

**Best-of-n** (BoN) sampling draws $n$ answers from $\pi_{\mathrm{ref}}$ and returns the one with
the highest proxy score. No weights change, yet BoN defines a new policy $\pi_{\mathrm{bon}}$, and
Gao et al. used it as the second way to optimize a proxy. Its distribution is exact. Sort the
answers by proxy score, ascending, and let $C_i = \sum_{j \le i} \pi_{\mathrm{ref}}(y_j)$. The best
of $n$ independent draws is at or below position $i$ exactly when all $n$ draws are, with
probability $C_i^n$, so

$$\pi_{\mathrm{bon}}(y_i) = C_i^{\,n} - C_{i-1}^{\,n}.$$

### Prediction

## Exercise 3 · the best-of-n distribution

`p_sorted` holds $\pi_{\mathrm{ref}}$ for answers sorted by ascending proxy score. Return
$\pi_{\mathrm{bon}}$ in the same order. Two lines: the cumulative sum $C$ (`p_sorted.cumsum(0)`),
then $C_i^n - C_{i-1}^n$ where $C_{i-1} = C_i - p_i$.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_bon_probs(p_sorted, n):
    C = ...                                         # C_i = Σ_{j ≤ i} p_j
    return ...                                      # C_i^n − C_{i−1}^n

In [ ]:
ex3_ok = s7.check_bon_probs(globals().get("my_bon_probs"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def bon_probs_ref(p_sorted, n):
    C = p_sorted.cumsum(0)
    return C ** n - (C - p_sorted).clamp(min=0) ** n    # clamp: C − p can round to −1e-17

s7.check_bon_probs(bon_probs_ref, strict=True)
bon_probs = mine_or_ref(globals(), "my_bon_probs", ex3_ok, bon_probs_ref)
print(inspect.getsource(s7.bon_policy))

### Reference explanation

### Reference answer and mechanism

$C_i^n$ is the probability that all $n$ draws land at or below rank $i$; subtracting
$C_{i-1}^n$ leaves the probability that the maximum is exactly answer $i$. With $n = 1$ this
returns $\pi_{\mathrm{ref}}$; as $n$ grows, the mass moves to the top of the ranking. `bon_policy`
sorts by proxy, applies your function and puts the probabilities back in the original order, so the
same `kl` and `s7.expect` work on it.

### Prediction

## Exercise 4 · the KL of best-of-n

When every answer has a small reference probability, $\pi_{\mathrm{bon}}(y) \approx n\,C^{\,n-1}\pi_{\mathrm{ref}}(y)$,
so $\log(\pi_{\mathrm{bon}}/\pi_{\mathrm{ref}}) \approx \log n + (n-1)\log C$. Under BoN, the chosen
answer's $C$ is the maximum of $n$ uniform draws, whose density is $n u^{n-1}$, and
$\mathbb{E}[\log C] = \int_0^1 n u^{n-1}\log u\,du = -1/n$. Write the resulting closed form. One line.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_bon_kl(n):
    return ...                                      # log n + (n − 1)·E[log C]

In [ ]:
ex4_ok = s7.check_bon_kl(globals().get("my_bon_kl"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def bon_kl_ref(n):
    return math.log(n) - (n - 1) / n

s7.check_bon_kl(bon_kl_ref, strict=True)
bon_kl = mine_or_ref(globals(), "my_bon_kl", ex4_ok, bon_kl_ref)
ns = [1, 2, 4, 8, 16, 32, 64, 128, 256, 1024, 4096]
bon = {"kl": [], "gold": [], "proxy": [], "oracle_gold": []}
print("   n | exact KL | log n − (n−1)/n | E[gold] (select by proxy) | E[gold] (select by gold)")
for n in ns:
    lp = s7.bon_policy(bon_probs, pool["logp_ref"], proxy, n)
    lo = s7.bon_policy(bon_probs, pool["logp_ref"], gold, n)       # oracle: select by the gold reward itself
    row = dict(kl=float(kl(lp, pool["logp_ref"])), gold=float(s7.expect(lp, gold)),
               proxy=float(s7.expect(lp, proxy)), oracle_gold=float(s7.expect(lo, gold)))
    for k in bon: bon[k].append(row[k])
    print(f"{n:4d} | {row['kl']:8.4f} | {bon_kl(n):15.4f} | {row['gold']:+25.3f} | {row['oracle_gold']:+.3f}")
bon = {k: torch.tensor(v) for k, v in bon.items()}

### Reference explanation

### Reference answer and mechanism

$\mathrm{KL} = \log n - (n-1)/n$. It grows only like $\log n$: best-of-4096 spends 7.3 nats. On this
pool, where each answer has probability $2^{-20}$, the formula matches the exact KL to four
decimals. Beirami et al. (2024) show that it is an upper bound in general; it overstates the KL when
$\pi_{\mathrm{ref}}$ puts large probability on a few answers (ties or a small answer set).

The table shows the overoptimization again, now without any training: selecting by proxy, gold
peaks around $n = 8$ (KL ≈ 1.2 nats) and is below the reference's gold by $n = 1024$. Selecting by
the gold reward itself (an oracle you never have) keeps improving.

### Prediction

## Predict 2 · best-of-n or the KL-regularized optimum?

At every KL value, $\pi_\beta$ has the highest expected **proxy** reward of all policies (that is
what maximizing $\mathbb{E}[r] - \beta\,\mathrm{KL}$ means). Best-of-8 spends 1.20 nats. Find the
$\pi_\beta$ with the same KL. Which of the two has the higher expected **gold** reward, and why?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
lo, hi = 0.3, 5.0                                   # bisection on β so that KL(π_β) = KL(best-of-8)
target = float(bon["kl"][ns.index(8)])
for _ in range(50):
    mid = (lo * hi) ** 0.5
    if float(kl(tilt(pool["logp_ref"], proxy, mid), pool["logp_ref"])) > target: lo = mid
    else: hi = mid
lp_beta = tilt(pool["logp_ref"], proxy, mid)
lp_bon = s7.bon_policy(bon_probs, pool["logp_ref"], proxy, 8)
for name, lp in (("π_β (β = %.3f)" % mid, lp_beta), ("best-of-8", lp_bon)):
    m, v = s7.expect(lp, proxy), s7.expect(lp, proxy ** 2) - s7.expect(lp, proxy) ** 2
    print(f"{name:18s} KL {kl(lp, pool['logp_ref']):.3f} | E[proxy] {m:.3f}, Var[proxy] {v:.3f} | "
          f"E[gold] {s7.expect(lp, gold):.3f} | mean length {s7.expect(lp, pool['length']):.0f} tokens")

### Reference explanation

### Reference answer and mechanism

**Best-of-8 has the higher gold reward** (about 0.65 against about 0.47), with a *lower* proxy
reward. Both policies reweight $\pi_{\mathrm{ref}}$ by a function of the proxy score alone, so their
gold reward depends only on the distribution of $r$ they induce, through
$m(r) = \mathbb{E}_{\mathrm{ref}}[g \mid r]$. For $a = 0.5, b = 1$ this conditional mean is
$m(r) = r/2 - r^2/8 + 1/4$, a concave function. Hence
$\mathbb{E}[g] = \mathbb{E}[r]/2 - (\mathrm{Var}[r] + \mathbb{E}[r]^2)/8 + 1/4$: spread in the proxy
costs gold. $\pi_\beta$ is a shifted Gaussian and keeps $\mathrm{Var}[r] = 2$; best-of-8 concentrates
on a narrow band of proxy values (variance about 0.75), and its answers are also a little shorter.

So "the most proxy per nat of KL" is not "the most gold per nat of KL". The two optimizers trace
different gold-vs-KL curves, as Gao et al. also found for BoN and RL; this toy explains the
difference only for its own concave $m(r)$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
x = np.log2(ns)
axes[0].plot(x, bon["proxy"], color=ORANGE, marker="o", markersize=3, label="proxy of chosen answer")
axes[0].plot(x, bon["gold"], color=BLUE, marker="o", markersize=3, label="gold, selecting by proxy")
axes[0].plot(x, bon["oracle_gold"], color=GREEN, marker="o", markersize=3, label="gold, selecting by gold (oracle)")
axes[0].axhline(0, color="black", linewidth=0.6)
axes[0].set(xlabel="log₂ n", ylabel="expected reward", title="best-of-n on the same pool")
axes[0].legend(fontsize=8)
c = curves[1.0]
m = c["kl"] <= 8
axes[1].plot(c["kl"][m].sqrt(), c["gold"][m], color=BLUE, label="KL-regularized optimum π_β")
axes[1].plot(bon["kl"].sqrt(), bon["gold"], color=ORANGE, marker="o", markersize=3, label="best-of-n")
axes[1].axhline(0, color="black", linewidth=0.6)
axes[1].set(xlabel="√KL from π_ref", ylabel="E[gold]", title="gold per unit of KL: BoN vs π_β")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Best-of-n proxy and gold reward against n, and gold against √KL for best-of-n and the KL-regularized optimum](../figures/chapter-14/study-path-s7_failure_modes-02.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part C · Length hacking and the length-controlled score

**Length hacking** is overoptimization through the length channel: the policy lengthens its answers
because the reward model pays for length, not because longer answers are better. Part A already
showed it: at KL ≈ 4 nats the mean answer is about three times as long. Two standard responses
follow from one estimate, the reward model's **length slope** under the reference policy,

$$\hat b = \frac{\mathrm{Cov}_{\pi_{\mathrm{ref}}}(r, z)}{\mathrm{Var}_{\pi_{\mathrm{ref}}}(z)},$$

the least-squares coefficient of the score on (log) length. The **length-controlled score**
$r_{\mathrm{LC}} = r - \hat b\,(z - \bar z)$ removes the linear length channel. Used as a
**reward**, it is a length penalty. Used as an **evaluation metric**, it compares two policies as if
their answers had the same length; AlpacaEval's length-controlled win rate (Dubois et al. 2024) is a
regression-based version of the same idea for pairwise judgments.

### Prediction

## Exercise 5 · the length slope and the length-controlled score

Given scores `r`, length scores `z` and reference weights `p` (all `[K]`), return
`(r_lc, b_hat)`. Use `p` for every mean and sum: $\bar z = \sum p z$, centered $z_c = z - \bar z$,
$\hat b = \sum p\, z_c\, r \,/\, \sum p\, z_c^2$, and $r_{\mathrm{LC}} = r - \hat b\, z_c$.
Three lines.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_length_controlled(r, z, p):
    zc = ...                                        # z minus its p-weighted mean
    b_hat = ...                                     # Cov_p(r, z) / Var_p(z)
    r_lc = ...                                      # remove the linear length channel
    return r_lc, b_hat

In [ ]:
ex5_ok = s7.check_length_controlled(globals().get("my_length_controlled"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def length_controlled_ref(r, z, p):
    zc = z - (p * z).sum()
    b_hat = (p * zc * r).sum() / (p * zc ** 2).sum()
    return r - b_hat * zc, b_hat

s7.check_length_controlled(length_controlled_ref, strict=True)
length_controlled = mine_or_ref(globals(), "my_length_controlled", ex5_ok, length_controlled_ref)
p_ref = pool["logp_ref"].exp()
r_lc, b_hat = length_controlled(proxy, pool["z"], p_ref)
print(f"estimated length slope b̂ = {b_hat:.4f} (true credit b = 1) | "
      f"Cov(r_lc, z) under π_ref = {(p_ref * (pool['z'] - (p_ref * pool['z']).sum()) * r_lc).sum():.1e}")
rewards = {"proxy r (credits length)": proxy,
           "r − ½·b̂·z  (half penalty)": proxy - 0.5 * b_hat * pool["z"],
           "r_LC = r − b̂·z  (full penalty)": r_lc}
length_curves = {name: s7.sweep(tilt, kl, pool["logp_ref"], R, betas, {"gold": gold, "length": pool["length"]})
                 for name, R in rewards.items()}
for name, c in length_curves.items():
    j = int((c["kl"] - 3).abs().argmin())
    print(f"{name:32s} at KL ≈ {c['kl'][j]:.1f}: mean length {c['length'][j]:4.0f} tokens, E[gold] {c['gold'][j]:+.3f}")

### Reference explanation

### Reference answer and mechanism

The slope uses the reference distribution, because that is where we can measure the reward model
before optimizing against it. Here $\hat b$ recovers the true length credit almost exactly (the
pool's $h$ and $z$ are independent), so $r_{\mathrm{LC}} = h + \text{const}$ and the policy
optimizing it never changes length: at the same KL its gold reward is far higher. The half penalty
lands in between.

Two limits apply outside the toy. The length channel of a real reward model need not be linear,
and the regression only removes the linear part. And if longer answers are partly *better* (D2:
HH's chosen answers are often more complete), $\hat b$ mixes legitimate and spurious credit, so a
full penalty also removes some real signal. Length control is a correction for one known channel,
not a test that a reward is clean.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for (name, c), color in zip(length_curves.items(), (RED, ORANGE, GREEN)):
    m = c["kl"] <= KL_MAX
    axes[0].plot(c["kl"][m], c["length"][m], color=color, label=name)
    axes[1].plot(c["kl"][m], c["gold"][m], color=color, label=name)
axes[0].axhline(float(pool["length"].mean()), color=GRAY, linestyle="--", linewidth=1)
axes[0].set(xlabel="KL(π_β ‖ π_ref)  (nats)", ylabel="mean answer length (tokens)", title="length of the optimum", yscale="log")
axes[1].axhline(0, color="black", linewidth=0.6)
axes[1].set(xlabel="KL(π_β ‖ π_ref)  (nats)", ylabel="E[gold]", title="gold of the optimum")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("Length-controlled evaluation: the same three policies scored three ways (change from π_ref)")
lc_mean = float((p_ref * r_lc).sum())
for beta in (2.0, 1.0, 0.5):
    lp = tilt(pool["logp_ref"], proxy, beta)                  # policies trained on the length-crediting proxy
    print(f"  π_β, β = {beta}: KL {kl(lp, pool['logp_ref']):.2f} | raw proxy {s7.expect(lp, proxy):+.3f} | "
          f"length-controlled {s7.expect(lp, r_lc) - lc_mean:+.3f} | gold {s7.expect(lp, gold):+.3f} | "
          f"length {s7.expect(lp, pool['length']):.0f} tokens")

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Mean answer length and gold reward of the KL-regularized optimum under the raw proxy, a half and a full length penalty](../figures/chapter-14/study-path-s7_failure_modes-03.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

The evaluation table shows what length control does to *comparisons*. Scored by the raw proxy,
the β = 0.5 policy gains +3.96 over the reference; the length-controlled score halves every gain
(+1.95), because half of each gain was length credit. But both scores rank the three policies in the
same order, β = 0.5 first, while the gold reward ranks β = 1 first and β = 0.5 last (below the
reference). The length-controlled score measures helpfulness at equal length; it cannot see the
damage that extreme lengths do, because that damage is not linear in length. A length-controlled
metric removes a known bias from a comparison; it is not a substitute for an independent judgment
of quality.

---

# Part D · Entropy collapse in a policy-gradient run

The exact optimum of Parts A–C hides one failure that only training shows. At a state $s$ (prompt
plus tokens so far), the policy's next-token distribution has **token entropy**

$$H(\pi(\cdot\mid s)) = -\sum_v \pi(v\mid s)\,\log\pi(v\mid s),$$

between 0 (one certain token) and $\log V$ (uniform over $V$ tokens). Training logs report its
mean over the generated tokens. **Entropy collapse** is that mean falling toward 0: the policy
becomes deterministic, stops sampling alternatives (exploration, glossary term 22), and every
answer in a group starts to look the same. Low entropy is not a failure by itself (a policy that
has solved a deterministic task should be confident); a fast fall *together* with rising reward
and KL is a warning sign.

### Prediction

## Exercise 6 · token entropy from logits

Return the entropy over the last dimension of `logits` (any shape `[..., V]`). One or two lines. Use
`torch.log_softmax` for $\log p$ and its `exp()` for $p$: `torch.log(torch.softmax(x))` returns
$-\infty$ where a probability underflows to 0, and $0 \cdot (-\infty)$ is `nan`.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_entropy(logits):
    return ...                                      # −Σ_v p_v log p_v over the last dimension

In [ ]:
ex6_ok = s7.check_entropy(globals().get("my_entropy"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def entropy_ref(logits):
    logp = torch.log_softmax(logits, -1)
    return -(logp.exp() * logp).sum(-1)

s7.check_entropy(entropy_ref, strict=True)
entropy = mine_or_ref(globals(), "my_entropy", ex6_ok, entropy_ref)
print(f"V = 151,936 (Qwen3 vocabulary): maximum entropy log V = {math.log(151936):.2f} nats")

### Reference explanation

### Reference answer and mechanism

`log_softmax` computes $x_v - \mathrm{logsumexp}(x)$, finite for every logit; where $p_v$
underflows, the product $p_v \log p_v$ is $0 \times (\text{finite}) = 0$, which is the correct
limit. Adding a constant to every logit leaves the entropy unchanged because softmax is
shift-invariant. For a real model, $H$ is computed at every response position from the same
forward pass that gives the token log-probabilities, so it costs one extra reduction over the
vocabulary.

### A tabular policy that can collapse

The smallest sequential policy: positions $t = 0, \dots, 15$, vocabulary $V = 6$ with token 0 =
**EOS** (end of answer), and a separate logit table per position, so the state is just the position.
$\pi_{\mathrm{ref}}$ stops with probability 0.2 per position (mean length ≈ 4 content tokens) and
is nearly uniform over the 5 content tokens.

- **Proxy reward**: each content token earns [0.5, 0.35, 0.2, 0.05, −0.1], summed over the answer.
  Longer answers earn more: a length channel.
- **Gold reward**: grades the tokens differently, [0.3, 0.45, 0.2, 0.05, −0.1], and subtracts
  $0.05\,(L-6)^2$ for answers longer than 6 tokens.

Because the policy is a table, the sequence KL and entropy are exact sums. Let
$\mathrm{reach}_t = \prod_{s<t}(1 - \pi_s(\mathrm{EOS}))$ be the probability that the answer is
still running at position $t$. By the chain rule,
$\mathrm{KL}(\pi\|\pi_{\mathrm{ref}}) = \sum_t \mathrm{reach}_t\,\mathrm{KL}(\pi_t\|\pi_{\mathrm{ref},t})$,
and the logged token entropy is the reach-weighted mean of $H(\pi_t)$.

Training is REINFORCE with a group baseline (S3): sample $N = 64$ answers, set
$A_i = R_i - \frac1N\sum_j R_j$ (the **advantage**: how much better answer $i$ scored than the
group), and minimize

$$\mathcal{L} = -\frac1N\sum_i A_i \log\pi(y_i) \;+\; \beta\,\mathrm{KL}(\pi\|\pi_{\mathrm{ref}}) \;-\; \lambda\,\bar H(\pi),$$

where $\log\pi(y_i)$ sums over the answer's tokens including EOS. The KL term is a **penalty** (it
raises the loss); the **entropy bonus** $\lambda\bar H$ lowers the loss when entropy is high.

### Prediction

## Exercise 7 · the policy-gradient loss with KL and entropy terms

Write $\mathcal{L}$ above. `logp_seq` and `adv` have shape `[N]`; `kl` and `ent` are scalars;
`beta` and `lam` are numbers. One line.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_pg_loss(logp_seq, adv, kl, ent, beta, lam):
    return ...                                      # −mean(A·log π) + β·KL − λ·H

In [ ]:
ex7_ok = s7.check_pg_loss(globals().get("my_pg_loss"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def pg_loss_ref(logp_seq, adv, kl, ent, beta, lam):
    return -(adv * logp_seq).mean() + beta * kl - lam * ent

s7.check_pg_loss(pg_loss_ref, strict=True)
pg_loss = mine_or_ref(globals(), "my_pg_loss", ex7_ok, pg_loss_ref)
print(inspect.getsource(s7.train_toy))
t0 = time.time()
toy_runs = {"β = 0, λ = 0": dict(beta=0.0, lam=0.0),
            "KL penalty β = 1": dict(beta=1.0, lam=0.0),
            "entropy bonus λ = 2": dict(beta=0.0, lam=2.0)}
toy_logs = {name: s7.train_toy(pg_loss, entropy, steps=300, **cfg) for name, cfg in toy_runs.items()}
ref_stats = s7.toy_stats(s7.toy_reference_logits(), s7.toy_reference_logits(), entropy)
print(f"3 runs × 300 steps in {time.time() - t0:.1f}s | π_ref: token entropy {ref_stats['token_entropy']:.3f} nats "
      f"(max log 6 = {math.log(6):.3f}), mean length {ref_stats['length']:.2f}")
print("step 1 → mean of the last 60 steps")
print("run                   | proxy       | gold        | KL (nats)   | token H     | length")
for name, log in toy_logs.items():
    f = lambda k: f"{log[k][0]:5.2f}→{s7.late(log[k]):5.2f}"
    print(f"{name:21s} | {f('reward')} | {f('gold')} | {f('kl')} | {f('entropy')} | {f('length')}")

### Reference explanation

### Reference answer and mechanism

The policy term has a minus sign because we minimize: lowering $\mathcal{L}$ raises
$\log\pi$ of answers with positive advantage. Its gradient with respect to each answer's
log-probability is $-A_i/N$. The KL term adds $+\beta$ per nat, the entropy bonus $-\lambda$ per
nat; a sign error on the bonus turns it into an entropy *penalty* and speeds up the collapse.

The table (step 1 → mean of the last 60 steps) shows three different outcomes from the same proxy:

- **β = 0**: the proxy rises from about 0.8 to about 7.9, the policy always writes 16 copies of the
  top proxy token, token entropy falls from 1.76 to below 0.1 nats, and KL reaches about 29 nats.
  Gold first rises a little and ends below where it started.
- **KL penalty β = 1**: entropy stays near the reference (≈ 1.7), KL stays under 1 nat and length
  grows only moderately. The penalty pulls every position back toward $\pi_{\mathrm{ref}}$, which
  has full support, so no position can become deterministic.
- **Entropy bonus λ = 2**: token entropy stays at about 0.9, but length still goes to 16 and KL to
  about 16 nats. The bonus keeps each position's distribution spread; it does nothing against the
  length channel. It treats a symptom, while the KL penalty limits how far the whole policy moves.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
colors = (RED, BLUE, GREEN)
for (name, log), color in zip(toy_logs.items(), colors):
    for ax, key in zip(axes, ("entropy", "length", "kl")):
        ax.plot(log[key], color=color, label=name)
axes[0].axhline(float(ref_stats["token_entropy"]), color=GRAY, linestyle="--", linewidth=1)
axes[0].set(xlabel="step", ylabel="token entropy (nats)", title="entropy")
axes[1].set(xlabel="step", ylabel="mean content tokens", title="answer length")
axes[2].set(xlabel="step", ylabel="KL(π ‖ π_ref)  (nats)", title="KL from the reference", yscale="log")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Token entropy, answer length and KL during tabular policy-gradient training without a penalty, with a KL penalty and with an entropy bonus](../figures/chapter-14/study-path-s7_failure_modes-04.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part E · Mitigations: what each one fixes

Three standard mitigations have now appeared: the **KL penalty** (choose β, that is, a point on
the gold-vs-KL curve), **length control** (Part C), and a third one, **reward-model ensembles**:
train $M$ reward models (different seeds or data subsets) and optimize an aggregate of their
scores. Coste et al. (2023) found that conservative aggregates, the minimum ("worst-case
optimization") or the mean minus a multiple of the standard deviation ("uncertainty-weighted"),
reduced overoptimization in a Gao-style setup. Eisenstein et al. (2023) found that ensembles
*mitigate but do not eliminate* reward hacking, because ensemble members share errors.

The toy makes both findings concrete. Give every answer two more independent features
$w_1, w_2 \sim \mathcal{N}(0,1)$. Reward model $k$ credits its own feature, an **independent
error**, and optionally the shared length feature, a **shared error**:

$$r_k = h + s\,b\,z + b\,w_k,\qquad g = h - a(z^2-1) - a(w_1^2-1) - a(w_2^2-1).$$

With $s = 0$ the two models' errors are independent; with $s = 1$ both also credit length.

### Prediction

## Exercise 8 · aggregating an ensemble of reward models

`scores` has shape `[M, K]` (one row per reward model). Return a `[K]` reward for three modes:
`"mean"`, `"min"` (the worst case over models) and `"lcb"` (mean − 1 · standard deviation over
models, population std: `unbiased=False`). One line each. Note that `torch.min(x, 0)` returns
`(values, indices)`.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_aggregate(scores, mode):                     # scores: [M models, K answers]
    if mode == "mean": return ...
    if mode == "min":  return ...
    if mode == "lcb":  return ...

In [ ]:
ex8_ok = s7.check_aggregate(globals().get("my_aggregate"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def aggregate_ref(scores, mode):
    if mode == "mean": return scores.mean(0)
    if mode == "min":  return scores.min(0).values
    if mode == "lcb":  return scores.mean(0) - scores.std(0, unbiased=False)
    raise ValueError(mode)

s7.check_aggregate(aggregate_ref, strict=True)
aggregate = mine_or_ref(globals(), "my_aggregate", ex8_ok, aggregate_ref)
betas_e = torch.logspace(1, -0.6, 36, dtype=torch.float64)
ens = {}
for shared in (0.0, 1.0):
    scores, gold_e = s7.ensemble_world(pool, M=2, b=1.0, shared=shared)
    for name, R in (("single RM", scores[0]), ("mean of 2", aggregate(scores, "mean")),
                    ("min of 2", aggregate(scores, "min"))):
        ens[shared, name] = s7.sweep(tilt, kl, pool["logp_ref"], R, betas_e, {"gold": gold_e})
print("errors        | reward     | peak gold (KL ≤ 6)  | gold at KL ≈ 1 | gold at KL ≈ 3")
for (shared, name), c in ens.items():
    m = c["kl"] <= 6
    i = int(c["gold"][m].argmax())
    at = lambda v: float(c["gold"][int((c["kl"] - v).abs().argmin())])
    print(f"{'independent' if shared == 0 else '+ shared len':13s} | {name:10s} | {c['gold'][m][i]:+.3f} at {c['kl'][m][i]:.2f} nats"
          f"   | {at(1):+.3f}         | {at(3):+.3f}")

### Reference explanation

### Reference answer and mechanism

The mean averages the errors; the minimum keeps, for every answer, the lower of the two scores. With
two models, mean − std is exactly the minimum: $\bar r - |r_1 - r_2|/2 = \min(r_1, r_2)$.

Read the table in two halves.

- **Independent errors only** ($s = 0$): a single reward model peaks early (≈ 0.49 at about 1 nat).
  The mean halves the credit each spurious feature receives, so gold peaks much later; the minimum
  is better still, because an answer scores high only if *both* models score it high, and the two
  models' errors rarely agree. Coste et al.'s finding, reproduced as a mechanism.
- **Plus a shared length credit** ($s = 1$): every aggregate still overoptimizes. Mean and min
  remove part of the independent error, but both models agree that longer is better, and no
  aggregate of the two can see an error they share. Eisenstein et al.'s finding: ensembles help in
  proportion to how independent their members' errors are.

In practice, members trained from the same pretrained model on the same preference data share most
of their errors, including length; the shared part needs a different fix (length control, better
data, a KL budget).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, shared, title in zip(axes, (0.0, 1.0), ("independent errors only", "independent errors + shared length credit")):
    for name, color in (("single RM", RED), ("mean of 2", ORANGE), ("min of 2", GREEN)):
        c = ens[shared, name]
        m = c["kl"] <= 8
        ax.plot(c["kl"][m].sqrt(), c["gold"][m], color=color, label=name)
    ax.axhline(0, color="black", linewidth=0.6)
    ax.set(xlabel="√KL(π_β ‖ π_ref)", title=title)
axes[0].set_ylabel("E[gold]")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Gold reward against √KL for a single reward model, the mean and the minimum of two, with independent and with shared errors](../figures/chapter-14/study-path-s7_failure_modes-05.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

| Mitigation | What it changes | What it cannot fix |
|---|---|---|
| KL penalty β | where on the gold-vs-KL curve training ends (Part A: β\* = 1 for b = 1) | the curve itself; the best β is unknown without a gold measurement |
| Length control (Part C) | removes the linear length channel; gold no longer falls through length | nonlinear length effects, other spurious features; may remove legitimate length credit |
| Ensemble mean / min (Part E) | shrinks errors that differ between members; min more than mean | errors all members share |
| Entropy bonus (Part D) | keeps per-token distributions spread | the reward's error; length and KL still grow |

---

# Part F · A diagnostic for training logs

In a real run there is no gold reward. What you can log every step is the proxy reward, the KL
from the reference, the token entropy and the response length. Three patterns in those four curves
are the classic warning signs:

1. **reward rising while KL has grown past a predeclared budget**: the policy is far from where the
   reward model was trained (Part A);
2. **length rising steadily**: the length channel is being used (Part C);
3. **entropy collapsing**: late entropy is a small fraction of early entropy (Part D).

To decide "rising" on noisy curves, use the **trend t-statistic**: the least-squares slope of the
series against the step number divided by its standard error. Values above about 3 are unlikely
under a flat series with independent noise (training curves are autocorrelated, so treat the
threshold as a screening rule, not a test). `s7.trend_t`, `s7.early` and `s7.late` (means of the
first and last 20% of steps) are given.

In [ ]:
print(inspect.getsource(s7.trend_t))

### Prediction

## Exercise 9 · flag the warning signs

Fill in the three flags. Each is one boolean expression using `s7.trend_t`, `s7.early`, `s7.late`
and the arguments. Keep the dictionary keys as given (the check reads them).

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_diagnose(log, kl_limit, ent_ratio=0.5, t_crit=3.0):
    return {
        "reward up while KL past limit": ...,       # reward trend above t_crit AND late KL above kl_limit
        "length rising": ...,                       # length trend above t_crit
        "entropy collapsing": ...,                  # late entropy below ent_ratio × early entropy
    }

In [ ]:
ex9_ok = s7.check_diagnose(globals().get("my_diagnose"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def diagnose_ref(log, kl_limit, ent_ratio=0.5, t_crit=3.0):
    return {
        "reward up while KL past limit": s7.trend_t(log["reward"]) > t_crit and s7.late(log["kl"]) > kl_limit,
        "length rising": s7.trend_t(log["length"]) > t_crit,
        "entropy collapsing": s7.late(log["entropy"]) < ent_ratio * s7.early(log["entropy"]),
    }

s7.check_diagnose(diagnose_ref, strict=True)
diagnose = mine_or_ref(globals(), "my_diagnose", ex9_ok, diagnose_ref)
for name, log in toy_logs.items():
    flags = diagnose(log, kl_limit=2.0)
    print(f"{name:21s} | trend t: reward {s7.trend_t(log['reward']):6.1f}, length {s7.trend_t(log['length']):6.1f} | "
          + (", ".join(k for k, v in flags.items() if v) or "no flags"))

### Reference explanation

### Reference answer and mechanism

On the Part D logs (KL budget 2 nats): the unpenalized run raises all three flags; the entropy-bonus
run raises the first two but not the entropy flag, which is exactly the blind spot of the bonus; the
KL-penalized run raises only "length rising" (its length did grow, from about 4 to about 9 tokens,
within its KL budget).

A flag is a reason to read samples and run a controlled check, not a diagnosis (Chapter 14 §14.8).
Length can rise because a task needs longer answers; entropy can fall because a task is solved. The
thresholds (KL budget, ratio 0.5, t = 3) are declared before looking at a run, so a flag is not
tuned to the result.

---

# Part G · A real run: Qwen3-0.6B, a reward model and GRPO with β = 0 and β > 0

Now the same measurements on real models. Three networks:

- **Reward model**: Qwen3-0.6B with its LM head replaced by a new `Linear(1024 → 1)` (D30),
  trained here on 1,000 HH helpful-base pairs with the Bradley–Terry loss
  $-\log\sigma(r_{\text{chosen}} - r_{\text{rejected}})$ (S1). Its score is read at the last
  token, `<|im_end|>` (D28). Then it is frozen.
- **Policy** $\pi_\theta$: Qwen3-0.6B (chat, thinking off), trained by GRPO (S4) against the
  frozen reward model.
- **Reference** $\pi_{\text{ref}}$: a frozen copy of the starting policy, for the KL term.

The RL prompts are HH user questions with one instruction appended: *"Please answer in two or three
sentences."* The base policy then writes about 60 tokens, far below the 128-token cap, so length has
room to grow, and growth beyond a few sentences also breaks the instruction. We run GRPO twice from
the same start, with the same prompts and seeds: **β = 0** (no KL penalty) and **β = 0.1**.

In [ ]:
tok = Q.load_tokenizer()
train_pairs, test_pairs = Q.hh_pairs("train"), Q.hh_pairs("test")
N_RM, RM_BS, RM_LR = 1000, 8, 1e-5
rm_train = random.Random(0).sample(train_pairs[:20000], N_RM)   # train_pairs[20000:] is never used by the RM
rm_eval = test_pairs[:300]
print(f"{len(train_pairs):,} train pairs, {len(test_pairs):,} test pairs | RM trains on {N_RM}, evaluates on {len(rm_eval)}")
print(inspect.getsource(Q.load_reward_model))
print(inspect.getsource(Q.rm_scores))

### Precision: fp32 master weights, bf16 compute

The two trained networks (reward model and policy) keep their weights in **float32** and run their
forward passes under `torch.autocast(..., dtype=torch.bfloat16)`; the frozen reference stays in
bf16. The reason is rounding. AdamW changes each weight by roughly the learning rate per step,
about $10^{-5}$ here. A bf16 number has 8 significant bits, so near a typical weight value of 0.02
the next representable bf16 number is about $1.2\times10^{-4}$ away: an update of $10^{-5}$ rounds
back to the same weight and is lost. In float32 the spacing near 0.02 is about $2\times10^{-9}$.

In [ ]:
for dtype, mantissa_bits in ((torch.bfloat16, 7), (torch.float32, 23)):
    w = torch.tensor(0.02, dtype=dtype)
    spacing = 2.0 ** (math.floor(math.log2(0.02)) - mantissa_bits)   # distance to the next representable number
    print(f"{str(dtype):15s}: 0.02 + 1e-5 == 0.02 ? {bool(w + torch.tensor(1e-5, dtype=dtype) == w)} | spacing near 0.02 = {spacing:.1e}")
AC = lambda: torch.autocast("cuda", dtype=torch.bfloat16)    # bf16 compute for fp32-weight models

In [ ]:
def score_batch(rm, prompts, answers):
    """Q.rm_scores without no_grad, for training: score at the last non-pad token."""
    texts = [Q.render(tok, p, a) for p, a in zip(prompts, answers)]
    enc = tok(texts, return_tensors="pt", padding=True, padding_side="right", truncation=True,
              max_length=320).to(DEVICE)
    with AC():
        return rm(**enc).logits.squeeze(-1).float()

rm = Q.load_reward_model(tok, dtype=torch.float32)            # fp32 master weights
opt = torch.optim.AdamW(rm.parameters(), lr=RM_LR, weight_decay=0.0)
rm.train(); rm_losses, t0 = [], time.time()
torch.cuda.reset_peak_memory_stats()
for i in range(0, N_RM, RM_BS):
    b = rm_train[i:i + RM_BS]
    s = score_batch(rm, [p["prompt"] for p in b] * 2, [p["chosen"] for p in b] + [p["rejected"] for p in b])
    loss = -F.logsigmoid(s[:len(b)] - s[len(b):]).mean()        # Bradley–Terry pairwise loss
    loss.backward(); torch.nn.utils.clip_grad_norm_(rm.parameters(), 1.0); opt.step(); opt.zero_grad()
    rm_losses.append(loss.item())
rm_peak = Q.gib(torch.cuda.max_memory_allocated())
del opt; Q.free()
rm.eval(); rm.requires_grad_(False)                             # frozen from here on: it only grades
print(f"{len(rm_losses)} steps in {time.time() - t0:.0f}s | mean loss: first 25 steps {np.mean(rm_losses[:25]):.3f}, "
      f"last 25 {np.mean(rm_losses[-25:]):.3f} (log 2 = 0.693) | peak GPU {rm_peak:.1f} GiB")

### Does this reward model reward length?

Three measurements on the 300 held-out pairs. **Pair accuracy**, with its binomial standard error.
The **correlation** between the score gap and the length gap (log of the token-length ratio), as in
Day 15 · 04, with a 95% interval from the Fisher transformation (one token is added to each length,
because a few HH answers are empty). And a direct comparison: in how
many pairs did the *humans* choose the longer answer, and in how many does the *reward model* rank
the longer answer higher? If the second number is larger, the reward model prefers length more
than the labels it learned from.

In [ ]:
prompts = [p["prompt"] for p in rm_eval]
with AC():
    sc = Q.rm_scores(rm, tok, prompts, [p["chosen"] for p in rm_eval]).cpu().double()
    sr = Q.rm_scores(rm, tok, prompts, [p["rejected"] for p in rm_eval]).cpu().double()
n_tok = lambda text: len(tok(text, add_special_tokens=False).input_ids)
lc = torch.tensor([n_tok(p["chosen"]) for p in rm_eval], dtype=torch.float64)
lr_ = torch.tensor([n_tok(p["rejected"]) for p in rm_eval], dtype=torch.float64)
n = len(rm_eval)
acc = float((sc > sr).double().mean())
print(f"held-out pair accuracy {acc:.3f} ± {math.sqrt(acc * (1 - acc) / n):.3f} (chance 0.5)")
gap, lgap = sc - sr, torch.log((lc + 1) / (lr_ + 1))          # +1: a few HH answers are empty
r = float(torch.corrcoef(torch.stack([gap, lgap]))[0, 1])
lo, hi = math.tanh(math.atanh(r) - 1.96 / math.sqrt(n - 3)), math.tanh(math.atanh(r) + 1.96 / math.sqrt(n - 3))
print(f"corr(score gap, log length ratio) = {r:+.2f}  (95% CI [{lo:+.2f}, {hi:+.2f}]; Day 15 · 04 measured +0.40 with 3,000 pairs)")
differ = lc != lr_
longer_chosen = float((lc > lr_)[differ].double().mean())
longer_scored = float(((sc > sr) == (lc > lr_))[differ].double().mean())
m = int(differ.sum())
print(f"pairs with different lengths: {m} | humans chose the longer answer: {longer_chosen:.3f} ± {math.sqrt(longer_chosen*(1-longer_chosen)/m):.3f}"
      f" | RM ranks the longer answer higher: {longer_scored:.3f} ± {math.sqrt(longer_scored*(1-longer_scored)/m):.3f}")

### Prediction

## Exercise 10 · mean token entropy over the response

For the real policy, `logits` has shape `[B, T, V]` (one next-token distribution per response
position) and `mask` has shape `[B, T]` (1 for response tokens through the first `<|im_end|>`, 0
for padding after it). Return the mean entropy over all response tokens in the batch: a
token-weighted mean, using your `entropy` from Exercise 6. One line.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_masked_entropy(logits, mask):
    return ...                                      # Σ (H · mask) / Σ mask

In [ ]:
ex10_ok = s7.check_masked_entropy(globals().get("my_masked_entropy"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def masked_entropy_ref(logits, mask):
    return (entropy(logits) * mask).sum() / mask.sum()

s7.check_masked_entropy(masked_entropy_ref, strict=True)
masked_entropy = mine_or_ref(globals(), "my_masked_entropy", ex10_ok, masked_entropy_ref)

SUFFIX = " Please answer in two or three sentences."
rl_questions = [q + SUFFIX for q in Q.hh_questions("train", start=20000)]
eval_questions = [q + SUFFIX for q in Q.hh_questions("test", start=600)[:32]]
policy = Q.load_chat_model(dtype=torch.float32)              # trained: fp32 master weights
ref = Q.load_chat_model(); ref.eval(); ref.requires_grad_(False)   # frozen: bf16
print(inspect.getsource(Q.rollout))
torch.manual_seed(0)
with AC():
    ro = Q.rollout(policy, tok, eval_questions[:16], max_new=128)
with torch.no_grad(), AC():
    logits = policy(input_ids=ro["ids"], attention_mask=ro["attn"], position_ids=ro["pos"]).logits[:, ro["P"] - 1:-1].float()
H_tok = entropy(logits)
print(f"base policy, 16 answers: {int(ro['mask'].sum())} response tokens, mean length {ro['mask'].sum(1).mean():.0f}, "
      f"truncated {int(ro['truncated'].sum())} | mean token entropy {masked_entropy(logits, ro['mask']):.3f} nats; "
      f"including padding positions it would read {H_tok.mean():.3f}")
print("Q:", eval_questions[0]); print("A:", Q.response_texts(tok, ro)[0])
del logits, H_tok

### Reference explanation

### Reference answer and mechanism

`entropy(logits)` has shape `[B, T]`; multiplying by the mask zeroes the padding positions, and
dividing by `mask.sum()` (the number of response tokens) gives a mean in which every generated token
counts once. Averaging per answer first would give short answers the same weight as long ones, a
different statistic (Chapter 14 §14.5 discusses the same choice for the loss). Including padding
positions measures distributions the policy never sampled from.

In the reference run the base policy's mean is about 1.2 nats per token, far below the maximum
$\log V \approx 11.9$: most positions are nearly determined by the context, a few are open.

### The GRPO step and the two runs

One GRPO iteration (S4): sample $G = 4$ answers for each of $P = 4$ prompts from the current
policy, score each with the frozen reward model, and set each answer's advantage to its group
z-score $A_i = (R_i - \bar R_{\text{group}})/(\mathrm{std}_{\text{group}} + 10^{-6})$. The loss is
the policy-gradient surrogate on every response token plus β times the mean per-token $k_3$ KL
estimate $e^{d} - d - 1$, $d = \log\pi_{\mathrm{ref}} - \log\pi_\theta$ (D35). One gradient step
per batch, so the importance ratio is exactly 1 and clipping is inactive. The log records, per
iteration: mean RM score, mean response length (tokens, including `<|im_end|>`), mean token entropy
(your Exercise 10), and the sequence KL estimate $\sum_t k_{3,t}$ averaged over the 16 answers.

Three shared helpers appear here for the first time. `Q.score_rollout(rm, ro)` runs the reward
model directly on the sampled token ids (policy and reward model share the tokenizer) and reads the
score at the last response token, like `Q.rm_scores` above. `Q.token_logps(ref, ro)` is
`policy_forward` without the logits: the reference's log-probability of each sampled token, with
logits at position $P-1+t$ predicting token $P+t$. `Q.free()` runs the garbage collector and empties
the CUDA cache after models are deleted.

Learning rate $5\times10^{-6}$ with fp32 master weights; with bf16 weights, as in Day 15 · 04, a
single AdamW step at $10^{-5}$ changed only about 10% of the policy's weights in a probe on the Spark,
so most of the intended update was lost (see the precision cell above). In fp32, 30 iterations
can move the policy measurably; the two runs differ only in β.

In [ ]:
P, G, MAX_NEW, N_IT, LR = 4, 4, 128, 30, 5e-6
BETAS = (0.0, 0.1)

def policy_forward(model, ro):
    """log π(a_t | s_t) of the sampled tokens and the full next-token logits, [B, T] and [B, T, V]."""
    with AC():
        logits = model(input_ids=ro["ids"], attention_mask=ro["attn"],
                       position_ids=ro["pos"]).logits[:, ro["P"] - 1:-1].float()
    logp = torch.log_softmax(logits, -1).gather(-1, ro["ids"][:, ro["P"]:, None]).squeeze(-1)
    return logp, logits

def grpo_step(policy, opt, questions, beta):
    with AC():
        ro = Q.rollout(policy, tok, [q for q in questions for _ in range(G)], max_new=MAX_NEW)
    m = ro["mask"]
    with torch.no_grad(), AC():
        R = Q.score_rollout(rm, ro)                          # [P·G] frozen reward-model scores
        ref_logp = Q.token_logps(ref, ro)
    logp, logits = policy_forward(policy, ro)
    Rg = R.view(-1, G)
    A = ((Rg - Rg.mean(1, keepdim=True)) / (Rg.std(1, keepdim=True) + 1e-6)).flatten()
    d = ref_logp - logp
    k3 = torch.exp(d) - d - 1                                # per-token KL estimate, ≥ 0
    pg = -(torch.exp(logp - logp.detach()) * A[:, None] * m).sum() / m.sum()
    loss = pg + beta * (k3 * m).sum() / m.sum()
    loss.backward(); torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0); opt.step(); opt.zero_grad()
    with torch.no_grad():
        H = masked_entropy(logits, m)
    return {"reward": R.mean().item(), "length": m.sum(1).mean().item(), "entropy": H.item(),
            "kl": (k3.detach() * m).sum(1).mean().item(), "truncated": ro["truncated"].float().mean().item()}

@torch.no_grad()
def samples(model, k=2, seed=7):
    """One sampled answer to each of the first k held-out questions, with its RM score."""
    torch.manual_seed(seed)
    with AC():
        ro = Q.rollout(model, tok, eval_questions[:k], max_new=MAX_NEW)
        R = Q.score_rollout(rm, ro).float()
    return list(zip(Q.response_texts(tok, ro), R.tolist(), ro["mask"].sum(1).tolist()))

@torch.no_grad()
def evaluate(model, seed=123, reps=2):
    """RM score and length of reps answers to each of the 32 fixed held-out questions: [32, reps] each."""
    torch.manual_seed(seed)
    with AC():
        ro = Q.rollout(model, tok, [q for q in eval_questions for _ in range(reps)], max_new=MAX_NEW)
        R = Q.score_rollout(rm, ro).float()
    return (R.view(-1, reps).cpu().double(), ro["mask"].sum(1).view(-1, reps).cpu().double())

base_eval = evaluate(policy)
real_samples = {"base": samples(policy)}
print(f"base policy on 32 held-out questions × 2: RM score {base_eval[0].mean():+.3f} ± {base_eval[0].mean(1).std() / 32**0.5:.3f}, "
      f"length {base_eval[1].mean():.1f} ± {base_eval[1].mean(1).std() / 32**0.5:.1f} tokens")

### Prediction

## Predict 3 · what will 30 iterations do?

Each run sees 30 × 16 = 480 sampled answers. The reward model's score gap correlates positively with
length (printed above). Predict, for β = 0 and for β = 0.1: does the mean RM score rise beyond its
noise? Does the mean response length rise? Which run ends with the larger KL? And would a higher RM
score show that the answers became more helpful?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
real_logs, real_evals = {}, {}
torch.cuda.reset_peak_memory_stats(); t_all = time.time()
for beta in BETAS:
    policy.load_state_dict(ref.state_dict())                     # identical start for both runs (bf16 values → fp32)
    opt = torch.optim.AdamW(policy.parameters(), lr=LR, weight_decay=0.0)
    rng = random.Random(1); torch.manual_seed(1)                 # identical prompt sequence
    log = {k: [] for k in ("reward", "length", "entropy", "kl", "truncated")}
    t0 = time.time()
    for it in range(N_IT):
        row = grpo_step(policy, opt, rng.sample(rl_questions, P), beta)
        for k, v in row.items(): log[k].append(v)
        if it % 10 == 9:
            print(f"β = {beta}: iter {it + 1:2d} | " + " | ".join(f"{k} {np.mean(v[-10:]):.3f}" for k, v in log.items())
                  + f" | {time.time() - t0:.0f}s")
    real_logs[beta] = log
    real_evals[beta] = evaluate(policy)
    real_samples[f"β = {beta}"] = samples(policy)
    del opt; Q.free()
grpo_peak = Q.gib(torch.cuda.max_memory_allocated())
print(f"both runs in {time.time() - t_all:.0f}s | peak GPU memory {grpo_peak:.1f} GiB")
print("\nheld-out evaluation, 32 questions × 2 answers; differences paired by question (mean ± standard error)")
for beta in BETAS:
    for k, name in ((0, "RM score"), (1, "length")):
        diff = real_evals[beta][k].mean(1) - base_eval[k].mean(1)
        print(f"  β = {beta}: {name:8s} {base_eval[k].mean():7.3f} → {real_evals[beta][k].mean():7.3f}  "
              f"(change {diff.mean():+.3f} ± {diff.std() / 32**0.5:.3f})")
for name, ev in (("base", base_eval), *((f"β = {b}", real_evals[b]) for b in BETAS)):
    print(f"  {name:8s}: fraction of held-out answers cut off at the {MAX_NEW}-token cap {(ev[1] >= MAX_NEW).double().mean():.2f}")

### Reference explanation

### Reference answer and mechanism

Read the numbers from your run; the reference run is described in the closing summary. What to
look for:

- **KL** is the most reliable signal: it starts at 0 (policy = reference) and grows, faster with
  β = 0. That shows the optimizer is moving the policy.
- **RM score and length** on the held-out questions: compare each change with its standard error.
  A change below about two standard errors is not evidence of an effect. With 480 samples per run,
  small or noisy changes are the expected outcome.
- Even a clear rise in RM score would **not** show more helpful answers: it is agreement with a
  roughly 60%-accurate proxy that also responds to length. Part A's curves are the reason.

In the reference run (the one that produced the saved figures):

- **β = 0**: the held-out length went from about 57 to 128 tokens (+70.5 ± 2.9), **every** answer
  ran into the 128-token cap, the KL estimate reached about 45 nats, and the RM score rose by
  +1.52 ± 0.19.
- **β = 0.1**: the length rose by +31 ± 2 tokens with 5% of answers at the cap, the KL stayed near
  10 nats, and the RM score rose by +0.75 ± 0.09.
- Token entropy did not collapse in either run (about 1.2–1.4 nats throughout).

A second run of the same notebook (same seeds; reward-model training and sampling on the GPU are
not bit-for-bit reproducible) gave the same β = 0 picture (+70.5 tokens, all answers at the cap,
RM +1.10 ± 0.24) but a smaller β = 0.1 drift (+11 ± 3 tokens, RM +0.07 ± 0.03). So the β = 0
result is robust; the size of the β = 0.1 effect is not well determined by one run. The next cells check where the RM gain came from.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for beta, color in zip(BETAS, (RED, BLUE)):
    log = real_logs[beta]
    for ax, key in zip(axes, ("reward", "length", "entropy", "kl")):
        y = np.array(log[key])
        ax.plot(y, color=color, alpha=0.3)
        ax.plot(np.arange(4, len(y)), np.convolve(y, np.ones(5) / 5, mode="valid"), color=color, label=f"β = {beta}")
for ax, title in zip(axes, ("RM score (batch mean)", "response length (tokens)", "token entropy (nats)", "KL per response (nats, Σ k3)")):
    ax.set(title=title, xlabel="GRPO iteration")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Real GRPO runs on Qwen3-0.6B against the trained reward model: RM score, response length, token entropy and KL per iteration for β = 0 and β = 0.1](../figures/chapter-14/study-path-s7_failure_modes-06.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

### Read the samples

Curves are summaries. The first thing to do when a run's reward or length moves is to read what the
policy now writes. The next cell prints one answer to each of two held-out questions from the base
policy and from both trained policies, with the reward model's score and the token count (the cap is
128 tokens; an answer at the cap was cut off).

In the reference run the β = 0 answers are two to four times longer than the base answers: fluent,
general elaboration ("Stories serve as a bridge between the past and present…") that runs into the
cap mid-sentence, ignoring the "two or three sentences" instruction in every prompt. The reward
model scored the first one 2.7 points above the base answer. The β = 0.1 answers are somewhat
longer than the base answers but end on their own. Read your own run's samples the same way: what changed in the text,
and would a person asking for two or three sentences prefer it?

In [ ]:
for k, q in enumerate(eval_questions[:2]):
    print("=" * 100); print("QUESTION:", q)
    for name, rows in real_samples.items():
        text, score, n_tok = rows[k]
        print(f"--- {name}: RM score {score:+.2f}, {int(n_tok)} tokens")
        print(text[:900] + (" […]" if len(text) > 900 else ""))

### Apply the diagnostic and the length-controlled score

The KL budget for the real run is declared here, before reading the flags: 5 nats per response.
The length-controlled comparison uses Exercise 5 on the *base* policy's held-out answers to estimate
the reward model's slope on log length, then subtracts that channel from every policy's scores.

In [ ]:
for beta in BETAS:
    flags = diagnose(real_logs[beta], kl_limit=5.0)
    log = real_logs[beta]
    print(f"β = {beta}: trend t — reward {s7.trend_t(log['reward']):+.1f}, length {s7.trend_t(log['length']):+.1f}, "
          f"entropy {s7.trend_t(log['entropy']):+.1f}; late KL {s7.late(log['kl']):.2f} nats | flags: "
          + (", ".join(k for k, v in flags.items() if v) or "none"))

z_base = torch.log(base_eval[1]).flatten()
r_base = base_eval[0].flatten()
_, b_real = length_controlled(r_base, z_base, torch.full_like(z_base, 1 / len(z_base)))
print(f"\nreward model's slope on log length among base answers: b̂ = {float(b_real):+.3f} score units per log-token")
z_mean = float(z_base.mean())
lc_score = lambda ev: (ev[0] - float(b_real) * (torch.log(ev[1]) - z_mean)).mean(1)
for beta in BETAS:
    raw = real_evals[beta][0].mean(1) - base_eval[0].mean(1)
    lcd = lc_score(real_evals[beta]) - lc_score(base_eval)
    print(f"β = {beta}: RM score change {raw.mean():+.3f} ± {raw.std() / 32**0.5:.3f} | "
          f"length-controlled change {lcd.mean():+.3f} ± {lcd.std() / 32**0.5:.3f}")

How to read these lines. A flag on a 30-iteration run is a reason to read samples, as in Part F.
A slope b̂ estimated from 64 base answers is noisy, the β = 0 answers lie far outside the base
length range (an extrapolation), and the correction removes legitimate length credit too (Part C).
The point of the comparison is the difference between the raw and the length-controlled change: if
the raw score rose and the length-controlled score did not, the rise came through length.

In the reference run:

- **β = 0** raised "reward up while KL past limit" (late KL about 46 nats against the declared 5)
  and "length rising". Its RM-score gain of +1.52 ± 0.19 shrinks to +0.85 ± 0.18 after length
  control: **about 45% of the measured gain is accounted for by the linear length channel**, and
  the rest is not (it may be other features the reward model credits, or real change; nothing here
  measures which). The correction is also an extrapolation: every β = 0 answer is longer than
  almost all base answers used to fit b̂.
- **β = 0.1** raised only the KL flag (late KL about 12 nats, also above the declared 5; its length
  trend over the run is not significant). Its RM gain of +0.75 ± 0.09 shrinks to +0.39 ± 0.09 after
  length control. In the second run the same numbers were +0.07 ± 0.03 and −0.00 ± 0.02.
- **No entropy collapse**: entropy stayed near 1.2–1.4 nats, so the entropy flag is correctly
  silent. Collapse (Part D) and length drift are different failures; this run shows the second.

In [ ]:
del policy, ref, rm; Q.free()
print(f"GPU memory still allocated: {Q.gib(torch.cuda.memory_allocated()):.2f} GiB")

---

# Summary

| Part | You built | Measured (toy numbers are exact for the stated rewards) |
|---|---|---|
| A | KL-regularized optimum `tilt`, exact `kl` | gold peaks at β\* = 1, KL ≈ 1 nat, gold ≈ 0.5 while the proxy keeps rising; a smaller length credit peaks later and higher |
| B | best-of-n distribution, BoN KL = log n − (n−1)/n | BoN gold peaks near n = 8; at equal KL best-of-8 has higher gold than π_β because gold is concave in the proxy |
| C | length slope and length-controlled score | removing the linear length channel keeps length fixed and gold rising with KL |
| D | token entropy, policy-gradient loss with KL and entropy terms | β = 0: entropy 1.76 → < 0.1 nats, length 4 → 16; KL penalty keeps entropy and KL small; entropy bonus keeps entropy but not length or KL |
| E | mean / min / lcb ensemble aggregation | min > mean > single RM with independent errors; all three still overoptimize when the members share a length credit |
| F | a three-flag training-log diagnostic | flags separate the three toy runs |
| G | masked token entropy for Qwen3 | 1,000-pair RM: 62% held-out accuracy, score-gap/length correlation +0.43 [0.33, 0.52], ranks the longer answer higher in 65% of pairs (humans chose it in 58%); GRPO β = 0: length 57 → 128 tokens (all at the cap), RM +1.52 ± 0.19, +0.85 ± 0.18 length-controlled; β = 0.1: length +31 ± 2, RM +0.75 ± 0.09 |

**Evidence.** The toy parts establish mechanisms exactly: for the stated gold and proxy rewards,
the KL-regularized optimum and best-of-n both overoptimize, the peak location follows from the
Gaussian formula, length control removes a linear channel, conservative ensembles help only against
independent errors, and an unpenalized policy-gradient run collapses its entropy while a KL penalty
does not. The real run shows, on Qwen3-0.6B with fp32 master weights and a reward model trained on
1,000 HH pairs (which itself prefers the longer answer more often than the human labels do), that
30 GRPO iterations without a KL penalty push every held-out answer to the 128-token cap, ignoring a
brevity instruction, while the RM score rises well beyond its standard error; with β = 0.1 the
length drift is smaller and the KL stays near 10 nats. Linear length control explains part of the
β = 0 gain, not all of it. What it does **not** show: any change in helpfulness (no gold judge was
used; the RM score is agreement with a roughly 60%-accurate proxy), a gold-reward curve like Part
A's (that needs an independent judge), or a well-determined size for the β = 0.1 effect (two runs
of the same notebook gave +31 and +11 tokens). For the
quantitative laws (how the gold peak scales with reward-model size and data), see Gao et al. 2022;
for ensembles, Coste et al. 2023 and Eisenstein et al. 2023.

**Next stage.** This is the last stage of the study path. Go back to the
[learning note](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) and record what you found: plan P2 (reward hacking) can now point to this
notebook, and D3 and D35 have toy numbers behind them. Chapter 14 §14.4–14.8 continues with the
rollout-system failures this notebook does not cover.